In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Methods: leakage and units — independent coding

Four small assignments. Try the learner version first. Reading a reference answer is not independent completion. All checks are illustrative tests, not a guarantee for every input.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/shape_lab').is_dir())
if str(ROOT / 'src') not in sys.path: sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shape_lab.evidence_v7 import load_snapshot, conformal_rank, conformal_radius, coverage, overlap_report
from shape_lab.display import show_and_save, save_json


## leakage.C1 — `shared_rows`

Return sorted row-index overlap.

Predict one result by hand before running the check.

<details><summary>Hint</summary>Set intersection answers row overlap only.</details>

In [3]:
def shared_rows(left,right):
    """Sorted intersection of row identities, not underlying participant identities."""
    return sorted(set(left)&set(right))


In [4]:
assert shared_rows([0,2],[1,3]) == []
print('leakage.C1: reference check passed')

leakage.C1: reference check passed


## leakage.C2 — `shared_units`

Return sorted shared unit IDs for two groups of rows.

Predict one result by hand before running the check.

<details><summary>Hint</summary>Resolve rows to source units before intersecting.</details>

In [5]:
def shared_units(left,right,units):
    """Underlying unit IDs may overlap even when row indices do not."""
    u=np.asarray(units)
    return sorted(set(u[left])&set(u[right]))


In [6]:
assert shared_units([0,2],[1,3],['A','A','B','B']) == ['A','B']
print('leakage.C2: reference check passed')

leakage.C2: reference check passed


## leakage.C3 — `partition_by_units`

Return row indices in selected units and in all remaining units.

Predict one result by hand before running the check.

<details><summary>Hint</summary>Build a membership mask; keep all copies together.</details>

In [7]:
def partition_by_units(units,selected):
    """Put all rows of each selected unit together; does not prove independence."""
    mask=np.isin(np.asarray(units),list(selected))
    return np.flatnonzero(mask),np.flatnonzero(~mask)


In [8]:
a,b=partition_by_units(['a','b','a','c'],['a'])
assert a.tolist()==[0,2] and b.tolist()==[1,3]
print('leakage.C3: reference check passed')

leakage.C3: reference check passed


## leakage.C4 — `apply_scaler`

Apply previously fitted feature centers and positive scales.

Predict one result by hand before running the check.

<details><summary>Hint</summary>Broadcast feature-wise arrays; do not fit anything inside this function.</details>

In [9]:
def apply_scaler(x,center,scale):
    """Apply fixed training parameters. Never refit them on validation or test."""
    x=np.asarray(x,float);c=np.asarray(center,float);s=np.asarray(scale,float)
    if x.ndim!=2 or c.shape!=(x.shape[1],) or s.shape!=c.shape or np.any(s<=0) or not np.isfinite(x).all() or not np.isfinite(c).all() or not np.isfinite(s).all():raise ValueError('Matching finite data and positive scales required')
    return (x-c)/s


In [10]:
assert np.allclose(apply_scaler([[6,2]],[2,2],[2,1]),[[2,0]])
print('leakage.C4: reference check passed')

leakage.C4: reference check passed
